In [ ]:
#!python -m spacy download en_core_web_trf

# Imports and configurations

In [8]:
import os
import pandas as pd
import spacy
import torch
import logging
from tqdm import tqdm
from rapidfuzz import process, fuzz
import re

In [9]:
# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s"
)

# Load spaCy model with GPU
spacy.require_gpu()
nlp = spacy.load("en_core_web_trf")

c:\Users\nkhan\Documents\github\news_collection\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Helper Functions

In [ ]:
# A comprehensive list of suffixes and stop words to remove for cleaner matching
REMOVAL_LIST = [
    # Suffixes
     'a.b.', 'a.g.', 'a.s.', 'a/s', 'ab', 'ads', 'ag', 'aktiengesellschaft', 'as',
    'asa', 'b.v.', 'bv', 'c.v.', 'co', 'co.', 'coltd', 'comp', 'company',
    'corp', 'corp.', 'corporation', 'corporations',  # Added 'corporations'
    'gmbh', 'inc', 'inc.', 'incorporated',
    'intl', 'k.k.', 'k.s.c.', 'kk', 'l.l.c.', 'l.p.', 'limited', 'llc', 'llp',
    'lp', 'ltd', 'ltd.', 'n.v.', 'oy', 'plc', 'pte', 's.a.', 's.a.r.l.',
    's.p.a.', 'sa', 'sarl', 'sas', 'se', 'spa', 'srl', 'class a', 'class b',
    'class c', 'non-voting', 'pref', 'group', 'hldgs', 'holdings',
    # Stop words
    'and', 'the', 'of'
]


# Helper function to clean organization names
def clean_name(name):
    """
    A more robust function to clean organization names. It handles hyphens,
    removes punctuation, handles possessives, and strips out suffixes and stop words.
    """
    if not isinstance(name, str):
        return ""
    
    name = name.lower()
    
    name = re.sub(r"\'s\b", " ", name) # Handle possessives ('s) 
    name = re.sub(r"\'\b", " ", name) # Handle possessives (trailing ')
    name = name.replace('-', ' ')  # Replace hyphens with spaces first
    name = re.sub(r'[^\w\s]', '', name)  # Remove remaining punctuation

    # Remove all words in the REMOVAL_LIST using word boundaries
    for word_to_remove in REMOVAL_LIST:
        name = re.sub(r'\b' + re.escape(word_to_remove) + r'\b', '', name)

    name = re.sub(r'\s+', ' ', name).strip()  # Clean up extra whitespace
    return name

# ALIAS_MAP: A dictionary to handle known variations and false negatives.
# Format: { cleaned_alias_name: cleaned_canonical_name }
# To be maintained carefully to ensure it only includes known variations

ALIAS_MAP = {

     # --- Rule 1: Add major rebrands and well-known aliases here ---
    clean_name("Google"): clean_name("ALPHABET INC"),
    clean_name("Facebook"): clean_name("META PLATFORMS INC"),
    clean_name("British Petroleum"): clean_name("BP PLC"),

    # --- Rule 2: Continue to add fixes for previously identified false negatives ---
    clean_name("Amazon.com"): clean_name("AMAZON COM INC"),
    clean_name("ExxonMobil"): clean_name("EXXON MOBIL CORP"),
    clean_name("Carlyle House"): clean_name("CARLYLE GROUP INC"),
    clean_name("Morgan Stanley Dean Witter"): clean_name("MORGAN STANLEY"),
    clean_name("Morgan Stanley Dean Witter & Company"): clean_name("MORGAN STANLEY"),
    clean_name("American Express TBS"): clean_name("AMERICAN EXPRESS"),
    clean_name("Amerada Hess"): clean_name("HESS CORP"),
    clean_name("LVMH Moet Hennessy"): clean_name("LVMH"),
    clean_name("LVMH Moet Hennessy Louis Vuitton"): clean_name("LVMH"),
    clean_name("Entergy Nuclear"): clean_name("ENTERGY CORP"),
    clean_name("Entergy Nuclear Northeast"): clean_name("ENTERGY CORP"),
    clean_name("Amphenol-Borg"): clean_name("AMPHENOL CORP CLASS A"), 
    clean_name("Estee Lauder Companies"): clean_name("ESTEE LAUDER INC CLASS A"),
    clean_name("Fox News"): clean_name("FOX CORP CLASS B"),
    clean_name("Fox News Channel"): clean_name("FOX CORP CLASS B"),
    clean_name("Fox News Network"): clean_name("FOX CORP CLASS B"),
    clean_name("20th Century Fox"): clean_name("FOX CORP CLASS B"),
    clean_name("Royal Dutch Shell"): clean_name("SHELL PLC"),
}



# === Setup MSCI reference ===
# ===== Clean MSCI World Data =====
def clean_msci_world(filepath):
    df = (
        pd.read_csv(filepath, skiprows=7, sep=';', decimal=',')
        .dropna(subset=['Ticker', 'Name'])
    )
    # Clean numeric columns
    num_cols = ['Market Value', 'Notional Value', 'Quantity', 'Price']
    for col in num_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace('.', '', regex=False)
                .str.replace(',', '.')
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')
    # Select and rename columns
    keep_cols = {
        'Ticker': 'ticker',
        'Name': 'name',
        'Market Value': 'market_value',
        'Notional Value': 'notional_value',
        'Location': 'location',
        'Sector': 'sector',
        'Asset Class': 'asset_class'
    }
    df = df[list(keep_cols.keys())].rename(columns=keep_cols)
    return df

# ==== Prepare MSCI Data ===
def prepare_lookup_structures(msci_df):
    """
    Applies the final cleaning function and prepares lookup structures.
    """
    msci_df['name_clean_processed'] = msci_df['name'].apply(clean_name)
    name_list = list(msci_df['name_clean_processed'].unique())
    name_lookup = dict(zip(msci_df['name_clean_processed'], msci_df[['name', 'ticker', 'location']].to_dict('records')))
    print("Lookup structures built with a cleaning function.")
    return name_lookup, name_list

def find_match(org, name_lookup, name_list, min_score=90):
    """
    Final matching function: checks aliases, then falls back to fuzzy matching.
    Returns a detailed dictionary with the outcome.
    """
    cleaned_org = clean_name(org)
    if not cleaned_org:
        return {"status": "DISCARD", "reason": "Source name empty", "source_org": org, "score": 0}

    # 1. Alias Lookup
    if cleaned_org in ALIAS_MAP:
        canonical_name = ALIAS_MAP[cleaned_org]
        if canonical_name in name_lookup:
            match_record = name_lookup[canonical_name]
            return {
                "status": "MATCH", "matched_org": org, "name": match_record["name"],
                "ticker": match_record["ticker"], "location": match_record["location"],
                "match_type": "Alias", "is_keyword_validated": True
            }

    # 2. Fuzzy Matching
    best_match_target, score, _ = process.extractOne(cleaned_org, name_list, scorer=fuzz.token_set_ratio)
    potential_match_record = name_lookup.get(best_match_target)

    if score < min_score:
        return {"status": "DISCARD", "reason": "Score too low", "source_org": org, "target_org": potential_match_record.get("name"), "score": score, "location": potential_match_record.get("location"), "ticker": potential_match_record.get("ticker")}

    source_keywords = set(cleaned_org.split())
    target_keywords = set(best_match_target.split())
    target_keywords_singular = {word[:-1] if word.endswith('s') else word for word in target_keywords}

    if not (source_keywords.issubset(target_keywords) or source_keywords.issubset(target_keywords_singular)):
        return {"status": "DISCARD", "reason": "Keyword mismatch", "source_org": org, "target_org": potential_match_record.get("name"), "score": score, "location": potential_match_record.get("location"), "ticker": potential_match_record.get("ticker")}

    return {
        "status": "MATCH", "matched_org": org, "name": potential_match_record["name"],
        "ticker": potential_match_record["ticker"], "location": potential_match_record["location"],
        "match_type": f"token_set_fuzzy ({score:.2f})", "is_keyword_validated": False
    }

# === Global Blacklist ===
GLOBAL_BLACKLIST = {
    "mets", "hip", "society", "state", "house", "gallery",
    "city", "bank", "academy", "university",
    "hospital", "center", "museum",
     "funeral" # Added based on review
}

def extract_orgs_batch(texts, batch_size=64):
    orgs_list = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        orgs = {ent.text.strip() for ent in doc.ents if ent.label_ == "ORG"}
        clean_orgs = []
        for org in orgs:
            org_lower = org.lower()
            if len(org) > 2 and org_lower not in GLOBAL_BLACKLIST:
                clean_orgs.append(org)
            elif len(org) > 2 and org_lower in GLOBAL_BLACKLIST and any(sub_org.lower().startswith(org_lower) and len(sub_org) > len(org) for sub_org in orgs if sub_org != org):
                pass 
            elif len(org) > 2 and org_lower in GLOBAL_BLACKLIST and not any(sub_org.lower().startswith(org_lower) and len(sub_org) > len(org) for sub_org in orgs if sub_org != org):
                pass 
            else:
                pass 
        orgs_list.append(clean_orgs)
    return orgs_list


In [11]:
# --- FINAL MAIN PIPELINE ---

def process_dataset_in_chunks(input_df, msci_df, save_path, discarded_save_path, output_format="parquet", chunk_size=50000, resume=False):
    """
    Final pipeline: processes data, creating matched and discarded files with consistent schemas
    and ensuring row_ids are not duplicated between them.
    """
    global name_lookup, name_list
    name_lookup, name_list = prepare_lookup_structures(msci_df)

    start_idx = 0
    processed = []
    discarded = []

    # Resume logic (no changes here)
    if resume and os.path.exists(save_path):
        if output_format == "csv":
            existing_processed = pd.read_csv(save_path)
            if os.path.exists(discarded_save_path):
                existing_discarded = pd.read_csv(discarded_save_path)
        else:
            existing_processed = pd.read_parquet(save_path)
            if os.path.exists(discarded_save_path):
                existing_discarded = pd.read_parquet(discarded_save_path)
        
        if not existing_processed.empty:
            start_idx = existing_processed["row_id"].max() + 1
            logging.info(f"Resuming from row {start_idx}")
            processed = existing_processed.to_dict("records")
            if 'existing_discarded' in locals() and not existing_discarded.empty:
                discarded = existing_discarded.to_dict("records")
        else:
             logging.info("Main output file is empty. Starting fresh.")
    else:
        logging.info("Starting fresh")


    for i in tqdm(range(int(start_idx), len(input_df), chunk_size), desc="Processing"):
        chunk = input_df.iloc[i:i + chunk_size].copy()
        texts = [f"{row.title}. {row.summary}" for _, row in chunk.iterrows()]
        orgs_batch = extract_orgs_batch(texts, batch_size=64)

        for row, orgs in zip(chunk.itertuples(), orgs_batch):
            # Temporary lists for the current row
            successful_matches_for_this_row = []
            discarded_matches_for_this_row = []

            for org in orgs:
                match_result = find_match(org, name_lookup, name_list)
                
                # Base record with full context from the original article
                base_record = {
                    "row_id": row.Index, "title": row.title, "summary": row.summary,
                    "source": row.source, "published_date": row.published_date, "keywords": row.keywords,
                }

                if match_result['status'] == 'MATCH':
                    # Update the base record with match-specific details
                    base_record.update({
                        "matched_org": match_result["matched_org"], "name": match_result["name"],
                        "ticker": match_result["ticker"], "location": match_result["location"],
                        "match_type": match_result["match_type"], "is_keyword_validated": match_result["is_keyword_validated"],
                    })
                    successful_matches_for_this_row.append(base_record)
                else:
                    # Create a discard record with the same rich schema
                    base_record.update({
                        "matched_org": match_result.get("source_org"), "name": match_result.get("target_org"),
                        "ticker": match_result.get("ticker"), "location": match_result.get("location"),
                        "match_type": f"Discard - {match_result.get('reason')} (Score: {match_result.get('score', 0):.2f})",
                        "is_keyword_validated": False,
                    })
                    discarded_matches_for_this_row.append(base_record)

            # NEW LOGIC: Decide where to put the results for this row
            if successful_matches_for_this_row:
                # If there's at least one good match, add all good matches to the main output
                processed.extend(successful_matches_for_this_row)
            elif discarded_matches_for_this_row:
                # Only if there are NO good matches, add all discards to the discard file
                discarded.extend(discarded_matches_for_this_row)
        
        # --- Save progress for both files ---
        if processed:
            df_out = pd.DataFrame(processed)
            if "keywords" in df_out.columns:
                df_out["keywords"] = df_out["keywords"].apply(lambda x: str(x) if not pd.isnull(x) else "[]")
            if output_format == "csv":
                df_out.to_csv(save_path, index=False)
            else:
                df_out.to_parquet(save_path, index=False)
            logging.info(f"Saved {len(processed)} successful matches to {save_path}")

        if discarded:
            df_discarded = pd.DataFrame(discarded)
            if output_format == "csv":
                df_discarded.to_csv(discarded_save_path, index=False)
            else:
                df_discarded.to_parquet(discarded_save_path, index=False)
            logging.info(f"Saved {len(discarded)} discarded attempts to {discarded_save_path}")

    logging.info("All done!")

# Load data

In [18]:
# Load articles
df = pd.read_csv('data/all_articles.csv')
df['source'] = df['url'].apply(lambda x: 'nytimes' if 'nytimes' in x else 'the guardian')
df['published_date'] = pd.to_datetime(df['published_date'], errors='coerce').dt.date

# Load MSCI data
msci_df = clean_msci_world('data/msci_world.csv') # File downloaded 27.05.2025 from https://www.ishares.com/us/products/239696/


Reason for chosing MSCI world index is because it contains 23 developed countries and not only focus on the US. 

In [19]:
process_dataset_in_chunks(
    input_df=df,
    msci_df=msci_df,
    save_path="data/matched_output_v6.parquet",  
    discarded_save_path="data/discarded_output_v6.parquet",
    output_format="parquet",  
    chunk_size=50000,
    resume=True
)

2025-06-19 12:31:54,439 — INFO — Starting fresh


Lookup structures built with a cleaning function.


Processing:   0%|          | 0/6 [00:00<?, ?it/s]2025-06-19 12:40:59,489 — INFO — Saved 1441 successful matches to data/matched_output_v6.parquet
2025-06-19 12:40:59,668 — INFO — Saved 62793 discarded attempts to data/discarded_output_v6.parquet
Processing:  17%|█▋        | 1/6 [09:05<45:26, 545.22s/it]2025-06-19 12:49:45,910 — INFO — Saved 3072 successful matches to data/matched_output_v6.parquet
2025-06-19 12:49:46,245 — INFO — Saved 117215 discarded attempts to data/discarded_output_v6.parquet
Processing:  33%|███▎      | 2/6 [17:51<35:37, 534.26s/it]2025-06-19 12:53:20,063 — INFO — Saved 10670 successful matches to data/matched_output_v6.parquet
2025-06-19 12:53:20,487 — INFO — Saved 146416 discarded attempts to data/discarded_output_v6.parquet
Processing:  50%|█████     | 3/6 [21:26<19:24, 388.13s/it]2025-06-19 12:57:00,688 — INFO — Saved 20479 successful matches to data/matched_output_v6.parquet
2025-06-19 12:57:01,181 — INFO — Saved 179359 discarded attempts to data/discarded_ou

In [1]:
import pandas as pd
# Load discarded data for review
discarded_df = pd.read_parquet("data/discarded_output_v6.parquet")
discarded_df.head()

,row_id,title,summary,source,published_date,keywords,matched_org,name,ticker,location,match_type,is_keyword_validated
0,3,"T. F. Lambert Jr., 85, Lawyer at Nuremberg","Thomas Francis Lambert Jr, law professor and d...",nytimes,2000-01-01,"['Lambert, Thomas Francis Jr', 'Biographical I...",Nuremberg,QUEBECOR INC CLASS B,QBR.B,Canada,Discard - Score too low (Score: 58.82),False
1,3,"T. F. Lambert Jr., 85, Lawyer at Nuremberg","Thomas Francis Lambert Jr, law professor and d...",nytimes,2000-01-01,"['Lambert, Thomas Francis Jr', 'Biographical I...",International Military Tribunal at Nuremberg,RPM INTERNATIONAL INC,RPM,United States,Discard - Score too low (Score: 86.67),False
2,6,"Paid Notice: Deaths BENEDETTI, ANTONIO A.","BENEDETTI-Antonio A.. Age 87, of Summit, New J...",nytimes,2000-01-01,"['BENEDETTI, ANTONIO A.']",the American Cancer Society,ANGLO AMERICAN PLC,AAL,United Kingdom,Discard - Score too low (Score: 72.73),False
3,6,"Paid Notice: Deaths BENEDETTI, ANTONIO A.","BENEDETTI-Antonio A.. Age 87, of Summit, New J...",nytimes,2000-01-01,"['BENEDETTI, ANTONIO A.']","the Bailey Funeral Home, Inc.",HOME DEPOT INC,HD,United States,Discard - Score too low (Score: 57.14),False
4,7,"Paid Notice: Deaths MACTAS, CELIA",MACTAS-Celia. Age 89. Survived by her loving c...,nytimes,2000-01-01,"['MacTas, Celia']",Robert Schoem's,ROPER TECHNOLOGIES INC,ROP,United States,Discard - Score too low (Score: 58.06),False


In [2]:
# Sample discarded data for review
sample_discarded = discarded_df.sample(100, random_state=42)
# Save sample for review
sample_discarded.to_csv("data/discarded_sample_v6.csv", index=False)

In [37]:
discarded_df_clean = discarded_df[discarded_df['match_type'].str.contains("Keyword mismatch")]
# Eliminate rows that contain "Paid Notice: Death" or "Funeral Home" in the title
discarded_df_clean = discarded_df_clean[~discarded_df_clean['title'].str.contains("Paid Notice: Death|Funeral Home", case=False, na=False)]
# Save cleaned discarded data for review as csv
discarded_df_clean.to_csv("data/discarded_output_v6_clean.csv", index=False)

In [ ]:
# Optional aliases
ORG_ALIASES = {
    "google": "alphabet",
    "meta": "meta platforms",
    "facebook": "meta platforms",
    "t-mobile": "tmobile",
    "p&g": "procter gamble",
    "procter & gamble": "procter gamble",
    "coca-cola": "coca-cola company", 
    "waltdisney": "walt disney company",
    "lvmh": "lvmh moet hennessy louis vuitton",
    "morgan stanley dean witter": "morgan stanley", # Added from previous review
    "american express tbs": "american express"
}
# === Entity Extraction ===
def extract_orgs_batch(texts, batch_size=64):
    orgs_list = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        orgs = {ent.text.strip() for ent in doc.ents if ent.label_ == "ORG"}
        clean_orgs = []
        for org in orgs:
            org_lower = org.lower()
            if len(org) > 2 and org_lower not in GLOBAL_BLACKLIST:
                # Add specific rules for filtering based on context (e.g., "Southern" used geographically)
            # This would require accessing the document, potentially too complex for this non-focus step
            # For "Southern", consider if "Southern" is part of a longer extracted name (e.g., "Southern California")

                clean_orgs.append(org)
            elif len(org) > 2 and org_lower in GLOBAL_BLACKLIST and any(sub_org.lower().startswith(org_lower) and len(sub_org) > len(org) for sub_org in orgs if sub_org != org):
                # If a blacklisted short org is part of a longer ORG (e.g., "Southern California"), keep the longer one.
                # This logic might need to be outside the list comprehension for better control.
                pass # For now, keep it simple and filter the short ones unless part of a compound.
            elif len(org) > 2 and org_lower in GLOBAL_BLACKLIST and not any(sub_org.lower().startswith(org_lower) and len(sub_org) > len(org) for sub_org in orgs if sub_org != org):
                pass # Discard short, blacklisted orgs that aren't part of a longer one
            else:
                pass # Discard very short (len <= 2) or generic
        orgs_list.append(clean_orgs)
    return orgs_list

# === ORG Matcher ===
# Adding 'keywords' and 'sector' to the match criteria
def match_single_org(org_text, name_lookup, name_list, article_keywords):
    org_clean = ORG_ALIASES.get(org_text.lower(), org_text.lower())
    
    best_match_candidate = None
    
    # Pre-process article keywords once for efficiency
    keywords_clean = [k.lower().replace(r'[^a-z0-9]', '') for k in article_keywords if isinstance(k, str)]

    # Tier 1: Exact Name or Ticker Match (HIGH CONFIDENCE - always accept)
    for name_msci_clean, msci_info in name_lookup.items():
        if org_clean == name_msci_clean or \
           (msci_info['ticker'] and org_clean == msci_info['ticker'].lower()):
            match = msci_info.copy()
            match["matched_org"] = org_text
            match["match_type"] = "exact_name_or_ticker"
            # No keyword validation needed for rejection at this tier, but log if it matches a keyword for analysis
            match["validated_by_keywords"] = any(name_part in kc for kc in keywords_clean for name_part in [org_clean, msci_info['name_keyword_clean']])
            return match
    
    # Tier 2: Strong token_set_ratio match (Requires keyword validation for acceptance)
    fuzzy_result = process.extractOne(org_clean, name_list, scorer=fuzz.token_set_ratio, score_cutoff=90) 
    if fuzzy_result:
        best_match_name_clean, score, idx = fuzzy_result
        msci_info = name_lookup[best_match_name_clean].copy()
        match = msci_info.copy()
        match["matched_org"] = org_text
        match["match_type"] = f"token_set_fuzzy ({score})"

        # Keyword Validation for Tier 2:
        # Check if matched company name, ticker, or its cleaned version is present in article's keywords
        is_keyword_relevant = False
        if keywords_clean:
            # Check if the full matched company name (cleaned) is in any keyword (cleaned)
            if msci_info['name_keyword_clean'] in keywords_clean:
                is_keyword_relevant = True
            # Check if the extracted ORG text itself is in any keyword (cleaned)
            elif org_clean.replace(r'[^a-z0-9]', '') in keywords_clean:
                is_keyword_relevant = True
            # Check if ticker is in keywords
            elif msci_info['ticker_clean'] and msci_info['ticker_clean'] in keywords_clean:
                is_keyword_relevant = True
            
        if is_keyword_relevant:
            match["validated_by_keywords"] = True
            return match
        else:
            # If Tier 2 and no keyword relevance, it's likely a false positive (e.g., "Southern" -> SOUTHERN)
            logging.info(f"Discarded potential match: '{org_text}' to '{match['name']}' due to keyword mismatch and not being a high-confidence exact match. Match Type: {match['match_type']}")
            return None
    else: 
        return None
    

# === Main Pipeline ===
def process_dataset_in_chunks(input_df, msci_df, save_path, output_format="parquet", chunk_size=10000, resume=False):
    global name_lookup, name_list
    name_lookup, name_list = prepare_msci(msci_df)

    start_idx = 0
    processed = []

    # Resume logic
    if resume and os.path.exists(save_path):
        if output_format == "csv":
            existing = pd.read_csv(save_path)
        else:
            existing = pd.read_parquet(save_path)
        start_idx = existing["row_id"].max() + 1
        logging.info(f"Resuming from row {start_idx}")
        processed = existing.to_dict("records")
    else:
        logging.info("Starting fresh")

    # Processing loop
    for i in tqdm(range(start_idx, len(input_df), chunk_size), desc="Processing"):
        chunk = input_df.iloc[i:i + chunk_size].copy()
        texts = [f"{row.title}. {row.summary}" for _, row in chunk.iterrows()]
        orgs_batch = extract_orgs_batch(texts, batch_size=64)

        for row, orgs in zip(chunk.itertuples(), orgs_batch):
            # Ensure keywords are treated as a list for consistency
            article_keywords = row.keywords if isinstance(row.keywords, list) else []
            if isinstance(article_keywords, str): # If keywords are a string representation of a list
                try:
                    article_keywords = eval(article_keywords) # Safely evaluate string to list
                except:
                    article_keywords = [] # Fallback if parsing fails

            for org in orgs:
                match = match_single_org(org, name_lookup, name_list, article_keywords)
                if match:
                    processed.append({
                        "row_id": row.Index,
                        "title": row.title,
                        "summary": row.summary,
                        "source": row.source,           
                        "published_date": row.published_date, 
                        "keywords": row.keywords, # Keep the original keywords for context
                        "matched_org": match["matched_org"],
                        "name": match["name"],
                        "ticker": match["ticker"],
                        "location": match["location"],
                        "match_type": match["match_type"],
                        "is_keyword_validated": match["validated_by_keywords"] # Add this for analysis
                    })
                    
        # Save progress
        df_out = pd.DataFrame(processed)
        # Convert 'keywords' column to JSON string for Parquet compatibility
        if "keywords" in df_out.columns:
            df_out["keywords"] = df_out["keywords"].apply(lambda x: str(x) if not pd.isnull(x) else "[]")

        if output_format == "csv":
            df_out.to_csv(save_path, index=False)
        else:
            df_out.to_parquet(save_path, index=False)
        logging.info(f"Saved {len(processed)} rows to {save_path} (up to row {i + chunk_size})")

    logging.info("All done!")

Matching 'Bristol-Myers Squibb' to 'BRISTOL MYERS SQUIBB':
  Cleaned Source: 'bristol myers squibb'
  Cleaned Target: 'bristol myers squibb'
  Result: MATCH (Match found, Score: 100.00)

Matching 'International Flavors and Fragrance' to 'INTERNATIONAL FLAVORS & FRAGRANCES':
  Cleaned Source: 'international flavors fragrance'
  Cleaned Target: 'international flavors fragrances'
  Result: NO MATCH (Keyword mismatch, Score: 98.41)



In [ ]:
# Check output
output_df = pd.read_parquet("data/matched_output.parquet")
print("\n--- Sample of Processed Data ---")
output_df.tail()

,row_id,title,summary,source,published_date,keywords,matched_org,name,ticker,location,match_type
0,24,"Paid Notice: Deaths GIBSON, LESLIE FULLER LARNED",GIBSON-Leslie Fuller Larned. Died December 14 ...,nytimes,2000-01-01,"['GIBSON, LESLIE FULLER LARNED']",Scenic Hudson,ENI,ENI,Italy,fuzzy (100.0)
1,35,Brooklyn Groups Back Use Of Park by Mets Farm ...,Brooklyn community groups drop their oppositio...,nytimes,2000-01-01,"['New York City', 'Prospect Park (NYC)', 'New ...",Mets,METSO CORPORATION,METSO,Finland,exact/alias
2,51,Tobacco and Its Money Have Minority Allies in ...,State Senator Efrain Gonzalez Jr. of the Bronx...,nytimes,2000-01-04,[],Philip Morris,PHILIP MORRIS INTERNATIONAL INC,PM,United States,exact/alias
3,52,"Paid Notice: Deaths STONE, MORTON D.","STONE-Morton D. died on January 2, 2000 at the...",nytimes,2000-01-04,"['STONE, MORTON D.']",HIP,CHIPOTLE MEXICAN GRILL INC,CMG,United States,exact/alias
4,80,"Paid Notice: Deaths GORDON, WILLIAM","GORDON - William. UNITE mourns Bill Gordon, a ...",nytimes,2000-01-04,"['Gordon, William']",UNITE,UNITEDHEALTH GROUP INC,UNH,United States,exact/alias
5,95,"Paid Notice: Deaths MCGEE, WILLIAM","MCGEE - William celebrated American painter, c...",nytimes,2000-01-04,"['McGee, William']",Borgenicht Gallery,ENI,ENI,Italy,fuzzy (100.0)
6,116,"Paid Notice: Deaths REESE, WILLIAM WILLIS",REESE-William Willis The Badminton Club of the...,nytimes,2000-01-04,"['REESE, WILLIAM WILLIS']",Sons of the Revolution,EVOLUTION,EVO,Sweden,fuzzy (100.0)
7,135,"Paid Notice: Deaths COYNE, JOHN J.",COYNE-John J. The Society of the Friendly Sons...,nytimes,2000-01-04,"['COYNE, JOHN J.']",Society,SOCIETE GENERALE SA,GLE,France,fuzzy (92.3076923076923)
8,149,Village Voice's Sale Turns Gadfly Into Chain F...,"The Village Voice, the 44-year-old alternative...",nytimes,2000-01-05,[],the Canadian Imperial Bank of Commerce,CANADIAN IMPERIAL BANK OF COMMERCE,CM,Canada,fuzzy (100.0)
9,179,"Suddenly, the Capitol Becomes Fort Knox","For 28 years, State Senator Frank Padavan has ...",nytimes,2000-01-06,[],State,ALLSTATE CORP,ALL,United States,exact/alias


## Add FNs back to list

In [ ]:
import pandas as pd
# Load reviewed discarded data
reviewed_discarded_df = pd.read_csv("results/discarded_output_v6_clean_reviewed.csv")
#reviewed_discarded_df = reviewed_discarded_df[reviewed_discarded_df['false_negative'] == 1]
#reviewed_discarded_df= reviewed_discarded_df.drop(columns=['false_negative', 'match_type'], errors='ignore')  # Drop review columns if they exist
#reviewed_discarded_df['match_type'] = 'reviewed_false_negative'  
#reviewed_discarded_df

,row_id,title,summary,source,published_date,keywords,matched_org,name,ticker,location,is_keyword_validated,match_type
2,115118,Connecting the Dots Isn���t Enough,Adobe Systems��� chief says that by setting a ...,nytimes,2009-07-18,"['ADOBE SYSTEMS INC', 'Executives and Manageme...",Adobe Systems���,ADOBE INC,ADBE,United States,False,reviewed_false_negative
4,176941,An Inspirational Interior for a Digital Designer,"For Geoff Dowd, director of experience design ...",nytimes,2015-04-18,"['Workplace Environment', 'Dowd, Geoffrey C (1...",Adobe Systems,ADOBE INC,ADBE,United States,False,reviewed_false_negative
5,238616,"John Warnock, Inventor of the PDF, Dies at 82","As a founder of Adobe Systems, he oversaw the ...",nytimes,2023-08-24,"['Warnock, John', 'Deaths (Obituaries)', 'Comp...",Adobe Systems,ADOBE INC,ADBE,United States,False,reviewed_false_negative
11,192383,"At Alcone Makeup Shop, Flesh and Blood. And Li...","Catering to beauty professionals, Alcone Compa...",nytimes,2016-12-02,"[""Hell's Kitchen (Manhattan, NY)"", 'Alcone Com...",Alcone Company,ALCON AG,ALC,Switzerland,False,reviewed_false_negative
12,243647,���Blade Runner 2049��� Producers Sue Elon Mus...,"Alcon Entertainment, the Hollywood company beh...",nytimes,2024-10-21,"['Movies', 'Electric and Hybrid Vehicles', 'Ta...",Alcon Entertainment,ALCON AG,ALC,Switzerland,False,reviewed_false_negative
...,...,...,...,...,...,...,...,...,...,...,...,...
5157,296355,Fossil fuel projects awaiting approval could b...,The Australian government will face decisions ...,the guardian,2023-11-30,"['Climate crisis', 'Energy', 'Greenhouse gas e...",Woodside Energy���s,WOODSIDE ENERGY GROUP LTD,WDS,Australia,False,reviewed_false_negative
5159,206271,WPP Advertising Agency Names Mark Read as Chie...,"Mr. Read, 51, is a longtime executive at the g...",nytimes,2018-09-03,"['Read, Mark (Marketing Executive)', 'WPP', 'A...",WPP Advertising Agency,WPP PLC,WPP,United Kingdom,False,reviewed_false_negative
5161,176226,Wynn Resorts Board Raises Objections to a Co-F...,A letter from the board of the casino company ...,nytimes,2015-03-24,"['Wynn, Elaine', 'Wynn Resorts Ltd', 'Boards o...",Wynn Resorts Board,WYNN RESORTS LTD,WYNN,United States,False,reviewed_false_negative
5162,176557,"Wynn Resorts��� Board, Its Nominees and Elaine...",The casino conglomerate���s corporate governan...,nytimes,2015-04-06,"['Wynn Resorts Ltd', 'Institutional Shareholde...",Wynn Resorts��� Board,WYNN RESORTS LTD,WYNN,United States,False,reviewed_false_negative


In [18]:
# Load output data
output_df = pd.read_parquet("data/matched_output_v6.parquet")
len(output_df)

48433

In [19]:
len(output_df) + len(reviewed_discarded_df)

49965

In [24]:
# Concat the reviewed false negatives  and the output DataFrame
final_output = pd.concat([output_df, reviewed_discarded_df], ignore_index=True)
final_output = final_output.drop_duplicates(subset=['row_id','name'], keep='last')
final_output

,row_id,title,summary,source,published_date,keywords,matched_org,name,ticker,location,match_type,is_keyword_validated
0,51,Tobacco and Its Money Have Minority Allies in ...,State Senator Efrain Gonzalez Jr. of the Bronx...,nytimes,2000-01-04,[],Philip Morris,PHILIP MORRIS INTERNATIONAL INC,PM,United States,token_set_fuzzy (100.00),False
1,122,"Paid Notice: Deaths SALTZMAN, RENNY B.",SALTZMAN-Renny B. Of New York and East Hampton...,nytimes,2000-01-04,"['SALTZMAN, RENNY B.']","Carlyle House, Inc.",CARLYLE GROUP INC,CG,United States,Alias,True
2,145,"Paid Notice: Deaths CRANDALL, ROLAND D.","CRANDALL-Roland D. Of Greenwich, Ct. on Januar...",nytimes,2000-01-04,"['CRANDALL, ROLAND D.']",Allen and Company Inc.,BOOZ ALLEN HAMILTON HOLDING CORP C,BAH,United States,token_set_fuzzy (100.00),False
3,149,Village Voice's Sale Turns Gadfly Into Chain F...,"The Village Voice, the 44-year-old alternative...",nytimes,2000-01-05,[],the Canadian Imperial Bank of Commerce,CANADIAN IMPERIAL BANK OF COMMERCE,CM,Canada,token_set_fuzzy (100.00),False
4,175,Final Federal Study Bolsters Case for Requirin...,Releasing the final installment in its eight-y...,nytimes,2000-01-05,[],G.E.,GE AEROSPACE,GE,United States,token_set_fuzzy (100.00),False
...,...,...,...,...,...,...,...,...,...,...,...,...
49960,296355,Fossil fuel projects awaiting approval could b...,The Australian government will face decisions ...,the guardian,2023-11-30,"['Climate crisis', 'Energy', 'Greenhouse gas e...",Woodside Energy���s,WOODSIDE ENERGY GROUP LTD,WDS,Australia,reviewed_false_negative,False
49961,206271,WPP Advertising Agency Names Mark Read as Chie...,"Mr. Read, 51, is a longtime executive at the g...",nytimes,2018-09-03,"['Read, Mark (Marketing Executive)', 'WPP', 'A...",WPP Advertising Agency,WPP PLC,WPP,United Kingdom,reviewed_false_negative,False
49962,176226,Wynn Resorts Board Raises Objections to a Co-F...,A letter from the board of the casino company ...,nytimes,2015-03-24,"['Wynn, Elaine', 'Wynn Resorts Ltd', 'Boards o...",Wynn Resorts Board,WYNN RESORTS LTD,WYNN,United States,reviewed_false_negative,False
49963,176557,"Wynn Resorts��� Board, Its Nominees and Elaine...",The casino conglomerate���s corporate governan...,nytimes,2015-04-06,"['Wynn Resorts Ltd', 'Institutional Shareholde...",Wynn Resorts��� Board,WYNN RESORTS LTD,WYNN,United States,reviewed_false_negative,False


In [28]:
# Save the final output
final_output["published_date"] = final_output["published_date"].astype(str)
final_output.to_parquet("data/01_org_matched_v6.parquet", index=False)

In [29]:
# 100 Sample of the final output for review
sample_output = final_output.sample(100, random_state=42)
sample_output.to_csv("data/samples/01_org_matched_v6_sample.csv", index=False)

# Old

In [147]:
# 2. Extract ORGs from title and summary
def extract_orgs(title, summary):
    text = f"{title}. {summary}"
    doc = nlp(text)
    orgs = {ent.text.strip() for ent in doc.ents if ent.label_ == "ORG"}
     # Filter out junk: single chars, hyphens, too short, etc.
    clean_orgs = {org for org in orgs if len(org) > 2 and org.lower() not in {"t", "-", "inc", "corp"}}
    return list(clean_orgs)

# 3. Match ORG list to MSCI World
def match_orgs_to_msci(org_list, msci_df, threshold=90):
    results = []
    names_clean = msci_df["name_clean"].tolist()


    for org in org_list:
        org_clean = org.lower().strip()
         # Apply alias if available
        alias_clean = ORG_ALIASES.get(org_clean, org_clean)

        # First try exact match
        matches = msci_df[msci_df["name_clean"].str.contains(alias_clean, regex=False)]
        if not matches.empty:
            match = matches.iloc[0][["name", "ticker", "location"]].to_dict()
            match["matched_org"] = org
            match['match_type'] = 'exact'
            results.append(match)
            continue
        
        # Fallback to fuzzy matching
        best_match, score, idx = process.extractOne(org_clean, names_clean, scorer=fuzz.partial_ratio)
        if score >= threshold:
            matched_row = msci_df.iloc[idx]
            match = matched_row[["name", "ticker", "location"]].to_dict()
            match["matched_org"] = org
            match["match_type"] = f"fuzzy ({score})"
            results.append(match)

    return results if results else None

# 4. Apply pipeline to your dataset
def process_row(row):
    orgs = extract_orgs(row["title"], row["summary"])
    matches = match_orgs_to_msci(orgs, msci_df)
    return pd.Series([orgs, matches])

# 6. Apply NER and matching
#tesla[["extracted_orgs", "matched_orgs"]] = tesla.progress_apply(process_row, axis=1)
google[["extracted_orgs", "matched_orgs"]] = google.progress_apply(process_row, axis=1)

# 7. Flatten: one row per matched organization
flat_records = []

for idx, row in google.iterrows():
    if row["matched_orgs"]:
        for match in row["matched_orgs"]:
            
            flat_records.append({
                "title": row["title"],
                "summary": row["summary"],
                "matched_org": match["matched_org"],
                "name": match["name"],
                "ticker": match["ticker"],
                "location": match["location"],
                "source": row["source"],
                "published_date": row["published_date"], 
                "keywords": row["keywords"]
            })

flat_df = pd.DataFrame(flat_records)
flat_df

100%|██████████| 2417/2417 [02:15<00:00, 17.80it/s]


,title,summary,matched_org,name,ticker,location,source,published_date,keywords
0,Questions and Praise for Google Web Library,"When Randall C. Jimerson, the president of the...",Google,ALPHABET INC CLASS A,GOOGL,United States,nytimes,2004-12-18,"['Google Inc', 'Libraries and Librarians', 'Co..."
1,Google Library Database Is Delayed,Senior product manager Adam Smith says Google ...,Google,ALPHABET INC CLASS A,GOOGL,United States,nytimes,2005-08-13,"['Harvard University', 'ASSOCIATION OF AMERICA..."
2,Googling Literature: The Debate Goes Public,If there was any point of agreement between pu...,Google,ALPHABET INC CLASS A,GOOGL,United States,nytimes,2005-11-19,"['Google Inc', 'Books and Literature', 'Comput..."
3,Mapping the Invisible City Outside Their Walls,"Dan Lloyd, philosophy department chairman at T...",Google,ALPHABET INC CLASS A,GOOGL,United States,nytimes,2006-05-03,"['Hartford (Conn)', 'Connecticut', 'TRINITY CO..."
4,Sunny and Gloomy Signs at a Web Crossroads,The main problem is that Yahoo has not been ne...,Google,ALPHABET INC CLASS A,GOOGL,United States,nytimes,2006-11-19,"['Yahoo Inc', 'Google Inc', 'Stocks and Bonds']"
...,...,...,...,...,...,...,...,...,...
2509,AI helps airline pilots avoid areas that creat...,Aircraft contrails – or clouds of condensation...,Microsoft,MICROSOFT CORP,MSFT,United States,the guardian,2023-08-09,"['Air pollution', 'Air transport', 'Google', '..."
2510,AI helps airline pilots avoid areas that creat...,Aircraft contrails – or clouds of condensation...,Boeing,BOEING,BA,United States,the guardian,2023-08-09,"['Air pollution', 'Air transport', 'Google', '..."
2511,Third of UK teenagers believe climate change e...,A third of UK teenagers believe climate change...,Google,ALPHABET INC CLASS A,GOOGL,United States,the guardian,2024-01-16,"['Climate crisis', 'UK news', 'Climate science..."
2512,Not a drop of common sense on bottled water,You report (9 December) that Harrogate Spring ...,Danone,DANONE SA,BN,France,the guardian,2024-12-10,"['Trees and forests', 'Environment', 'Water', ..."


In [7]:
# Helper functions
def batch_extract_orgs(rows, batch_size=32):
    texts = [f"{title}. {summary}" for title, summary in zip(rows["title"], rows["summary"])]
    orgs_list = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        orgs = {ent.text.strip() for ent in doc.ents if ent.label_ == "ORG"}
        # Filter garbage
        clean_orgs = [org for org in orgs if len(org) > 2 and org.lower() not in {"t", "-", "inc", "corp"}]
        orgs_list.append(clean_orgs)
    return orgs_list

def match_single_org(org, threshold=90):
    org_clean = ORG_ALIASES.get(org.lower(), org.lower())

    # First try exact or substring
    for name_clean in name_list:
        if org_clean in name_clean:
            match = name_lookup[name_clean].copy()
            match["matched_org"] = org
            match["match_type"] = "exact/alias"
            return match

    # Fuzzy fallback
    best_match, score, idx = process.extractOne(org_clean, name_list, scorer=fuzz.partial_ratio)
    if score >= threshold:
        match = name_lookup[name_list[idx]].copy()
        match["matched_org"] = org
        match["match_type"] = f"fuzzy ({score})"
        return match

    return None

In [8]:
def process_dataset_in_chunks(input_df, save_path, chunk_size=10000, resume=False):
    start_idx = 0
    processed = []

    if resume and os.path.exists(save_path):
        existing = pd.read_parquet(save_path)
        start_idx = existing["row_id"].max() + 1
        print(f"🔄 Resuming from row {start_idx}")
        processed = existing.to_dict("records")
    else:
        print("🚀 Starting from scratch")

    for i in range(start_idx, len(input_df), chunk_size):
        chunk = input_df.iloc[i:i + chunk_size].copy()
        texts = [f"{row.title}. {row.summary}" for _, row in chunk.iterrows()]
        orgs_batch = extract_orgs_batch(texts, batch_size=64)

        for row, orgs in zip(chunk.itertuples(), orgs_batch):
            for org in orgs:
                match = match_single_org(org)
                if match:
                    processed.append({
                        "row_id": row.Index,
                        "title": row.title,
                        "summary": row.summary,
                        "matched_org": match["matched_org"],
                        "name": match["name"],
                        "ticker": match["ticker"],
                        "location": match["location"],
                        "match_type": match["match_type"],
                        "source": row.source,
                        "published_date": row.published_date,
                        "keywords": row.keywords
                    })

        # Save progress every chunk
        pd.DataFrame(processed).to_parquet(save_path, index=False)
        print(f"Saved {len(processed)} records to {save_path} (up to row {i + chunk_size})")

    print("Done!")


In [9]:
msci_df

,ticker,name,market_value,notional_value,location,sector,asset_class,name_clean,ticker_clean
0,NVDA,NVIDIA CORP,2.113071e+08,2.113071e+08,United States,Information Technology,Equity,nvidia corp,nvda
1,MSFT,MICROSOFT CORP,2.089668e+08,2.089668e+08,United States,Information Technology,Equity,microsoft corp,msft
2,AAPL,APPLE INC,1.929814e+08,1.929814e+08,United States,Information Technology,Equity,apple inc,aapl
3,AMZN,AMAZON COM INC,1.250029e+08,1.250029e+08,United States,Consumer Discretionary,Equity,amazon com inc,amzn
4,META,META PLATFORMS INC CLASS A,8.975737e+07,8.975737e+07,United States,Communication,Equity,meta platforms inc class a,meta
...,...,...,...,...,...,...,...,...,...
1377,Z M5,FTSE 100 INDEX JUN 25,0.000000e+00,1.411302e+06,--,Cash and/or Derivatives,Futures,ftse 100 index jun 25,z m5
1378,VGM5,EURO STOXX 50 JUN 25,0.000000e+00,1.993138e+06,European Union,Cash and/or Derivatives,Futures,euro stoxx 50 jun 25,vgm5
1379,ESM5,S&P500 EMINI JUN 25,0.000000e+00,1.308825e+07,--,Cash and/or Derivatives,Futures,s&p500 emini jun 25,esm5
1380,MARGIN_GBP,FUTURES GBP MARGIN BALANCE,-4.291900e+03,-4.291900e+03,United Kingdom,Cash and/or Derivatives,Cash Collateral and Margins,futures gbp margin balance,margin_gbp


In [11]:
# Precompile name list
name_lookup = dict(zip(msci_df["name_clean"], msci_df[["name", "ticker", "location"]].to_dict("records")))
name_list = list(name_lookup.keys())

In [152]:
import logging
logging.basicConfig(level=logging.INFO)